In [ ]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from sentence_transformers import SentenceTransformer
import chromadb
from tqdm import tqdm

dir_loader = DirectoryLoader(
    r"..\AutoPreprocessor\data",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)
chunks = dir_loader.load()
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [chunk.page_content.lower().strip() for chunk in tqdm(chunks, desc="Preprocessing Text")]
embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True,     
    batch_size=32
)
client = chromadb.PersistentClient(path=r"..\AutoPreprocessor\chroma_db")
try:
    client.delete_collection("rag_collection")
except:
    pass
collection = client.get_or_create_collection(
    name="rag_collection",
    metadata={"hnsw:space": "cosine"}
)
ids = [
    chunk.metadata["source"].split("\\")[-1].replace(".txt","")
    for chunk in tqdm(chunks, desc="Preparing IDs")
]
metadatas = [
    {
        "source": chunk.metadata.get("source", ""),
        "encoding": "ordinal"
    }
    for chunk in tqdm(chunks, desc="Preparing Metadata")
]
if collection.count() == 0:
    collection.add(
        ids=ids,
        documents=texts,
        embeddings=embeddings.tolist(),
        metadatas=metadatas
)
